# Velora — Build + Simulation
Upload the GitHub repository ZIP. This notebook builds `Velora-debug.apk` and, when the Colab runtime exposes KVM, launches a headless Android emulator and captures the main Velora Simulation Mode screens.

## 1. Upload the GitHub ZIP
On GitHub choose **Code → Download ZIP**, then run the next cell and upload that ZIP.

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_name = next(name for name in uploaded if name.lower().endswith('.zip'))
print('Using:', zip_name)

## 2. Extract and detect the Android project

In [ ]:
import os, pathlib, shutil, subprocess, zipfile, time
work = pathlib.Path('/content/velora-build')
if work.exists(): shutil.rmtree(work)
work.mkdir(parents=True)
with zipfile.ZipFile('/content/' + zip_name) as z: z.extractall(work)
candidates = list(work.rglob('settings.gradle.kts')) + list(work.rglob('settings.gradle'))
if not candidates: raise RuntimeError('No Gradle Android project found in ZIP')
project = min((p.parent for p in candidates), key=lambda p: len(p.parts))
print('Project root:', project)

## 2.5 Verify Velora source version before installing/building

In [ ]:
import re, pathlib

build_file = project / 'app' / 'build.gradle.kts'
build_text = build_file.read_text()
version_name = re.search(r'versionName\s*=\s*"([^"]+)"', build_text)
version_code = re.search(r'versionCode\s*=\s*(\d+)', build_text)
print('NOTEBOOK BUILD       : beta12-gesture-adaptive')
print('Velora source version:', version_name.group(1) if version_name else 'unknown')
print('Velora versionCode    :', version_code.group(1) if version_code else 'unknown')

MIN_VERSION_CODE = 33
resolved_version_code = int(version_code.group(1)) if version_code else 0
if resolved_version_code < MIN_VERSION_CODE:
    raise RuntimeError(f'Velora ZIP is stale: versionCode {resolved_version_code}; expected at least {MIN_VERSION_CODE}. Download the latest GitHub ZIP.')

known_blockers = {
    'old zen settings action': 'Settings.ACTION_ZEN_MODE_SETTINGS',
    'invalid edit-bar padding': '.padding(horizontal = 16.dp, bottom = 88.dp)',
    'JVM clash: transition mode setter': 'fun setHomeTransitionMode(',
    'JVM clash: transition softness setter': 'fun setTransitionSoftness(',
}

source_files = list((project / 'app' / 'src').rglob('*.kt'))
hits = []
for label, needle in known_blockers.items():
    for file in source_files:
        text = file.read_text(errors='ignore')
        if needle in text:
            hits.append((label, file.relative_to(project)))

if hits:
    print('\nSTALE / KNOWN-BROKEN SOURCE DETECTED:')
    for label, file in hits:
        print(' -', label, '->', file)
    raise RuntimeError('Uploaded ZIP contains an older Velora source tree. Download the latest GitHub ZIP and rerun from a clean extraction.')

print('Source preflight: PASS')


## 3. Install JDK + Android SDK

In [ ]:
import os, pathlib, shutil, subprocess
subprocess.run('apt-get update -qq', shell=True, check=True)
subprocess.run('DEBIAN_FRONTEND=noninteractive apt-get install -y -qq openjdk-17-jdk wget unzip ca-certificates', shell=True, check=True)

sdk = pathlib.Path('/content/android-sdk')
cmdline = sdk / 'cmdline-tools' / 'latest'
sdkmanager_path = cmdline / 'bin' / 'sdkmanager'

if not sdkmanager_path.exists():
    archive = pathlib.Path('/content/cmdline-tools.zip')
    temp = pathlib.Path('/content/cmdline-temp')
    subprocess.run(['wget','-q','https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip','-O',str(archive)], check=True)
    if temp.exists(): shutil.rmtree(temp)
    temp.mkdir(parents=True)
    subprocess.run(['unzip','-q',str(archive),'-d',str(temp)], check=True)
    cmdline.parent.mkdir(parents=True, exist_ok=True)
    if cmdline.exists(): shutil.rmtree(cmdline)
    shutil.move(str(temp / 'cmdline-tools'), str(cmdline))
    shutil.rmtree(temp, ignore_errors=True)

os.environ['ANDROID_SDK_ROOT'] = str(sdk)
os.environ['ANDROID_HOME'] = str(sdk)
os.environ['PATH'] = str(cmdline / 'bin') + ':' + str(sdk / 'platform-tools') + ':' + str(sdk / 'emulator') + ':' + os.environ['PATH']

sdkmanager = shutil.which('sdkmanager') or str(sdkmanager_path)
if not pathlib.Path(sdkmanager).exists():
    raise RuntimeError('sdkmanager was not installed correctly')

pathlib.Path(sdkmanager).chmod(pathlib.Path(sdkmanager).stat().st_mode | 0o111)
subprocess.run('yes | sdkmanager --licenses >/dev/null', shell=True, check=False)
subprocess.run(['sdkmanager','platform-tools','platforms;android-35','build-tools;35.0.0'], check=True)

(project / 'local.properties').write_text('sdk.dir=' + str(sdk) + '\n')
print('Android SDK ready:', sdk)
subprocess.run('sdkmanager --list_installed | grep -E "platform-tools|android-35|35.0.0"', shell=True, check=False)


# Build signed phone-test release
This notebook now builds only one installable artifact: `Velora-release.apk`. No Google Drive, no JKS upload, no debug APK, and no emulator step.

In [ ]:
import hashlib, os, pathlib, shutil, subprocess
from getpass import getpass

gradle_home = pathlib.Path('/content/gradle-8.9')
if not gradle_home.exists():
    gradle_zip = pathlib.Path('/content/gradle-8.9-bin.zip')
    subprocess.run(['wget','-q','https://services.gradle.org/distributions/gradle-8.9-bin.zip','-O',str(gradle_zip)], check=True)
    subprocess.run(['unzip','-q',str(gradle_zip),'-d','/content'], check=True)

gradle = gradle_home / 'bin' / 'gradle'
gradle.chmod(gradle.stat().st_mode | 0o111)

env = os.environ.copy()
env['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
env['ANDROID_HOME'] = '/content/android-sdk'
env['ANDROID_SDK_ROOT'] = '/content/android-sdk'
env['PATH'] = '/content/android-sdk/cmdline-tools/latest/bin:/content/android-sdk/platform-tools:' + env['PATH']

BUILD_CHANNEL = 'sideload'  # 'sideload' for phone testing; use 'play' only for the Play distribution
if BUILD_CHANNEL not in {'sideload', 'play'}:
    raise RuntimeError("BUILD_CHANNEL must be 'sideload' or 'play'")

password = getpass('Velora release password: ')
if len(password) < 6:
    raise RuntimeError('Release password must be at least 6 characters.')

keytool = shutil.which('keytool')
if not keytool:
    raise RuntimeError('keytool not found')

keystore = pathlib.Path('/content/velora-release.jks')
if keystore.exists():
    keystore.unlink()

subprocess.run([
    keytool, '-genkeypair', '-v',
    '-storetype', 'JKS',
    '-keystore', str(keystore),
    '-storepass', password,
    '-keypass', password,
    '-alias', 'velora',
    '-keyalg', 'RSA',
    '-keysize', '2048',
    '-validity', '10000',
    '-dname', 'CN=Velora Launcher, OU=Velora, O=Velora, C=IN',
], check=True)

env_release = env.copy()
env_release['VELORA_KEYSTORE'] = str(keystore)
env_release['VELORA_STORE_PASSWORD'] = password
env_release['VELORA_KEY_ALIAS'] = 'velora'
env_release['VELORA_KEY_PASSWORD'] = password

variant_name = BUILD_CHANNEL[:1].upper() + BUILD_CHANNEL[1:] + 'Release'
gradle_task = ':app:assemble' + variant_name
print('Building Velora', BUILD_CHANNEL, 'release APK...')
print('Gradle task:', gradle_task)
if BUILD_CHANNEL == 'play':
    print('NOTE: Play release declares Notification Listener for Play-distributed builds; do not use it as the normal sideload test APK.')
print('Gradle output will stream below. If it fails, Velora will download the compiler log automatically.')
process = subprocess.Popen(
    [str(gradle), '--no-daemon', '--stacktrace', gradle_task],
    cwd=project,
    env=env_release,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
gradle_lines = []
for line in process.stdout:
    print(line, end='')
    gradle_lines.append(line)
gradle_exit = process.wait()
if gradle_exit != 0:
    failure_log = pathlib.Path('/content/Velora-gradle-failure.txt')
    failure_log.write_text(''.join(gradle_lines))
    print('\n--- VELORA BUILD FAILED: LAST 220 GRADLE LINES ---')
    print(''.join(gradle_lines[-220:]))
    print('Full log:', failure_log)
    try:
        from google.colab import files as colab_files
        colab_files.download(str(failure_log))
    except Exception as download_error:
        print('Could not auto-download failure log:', download_error)
    raise RuntimeError('Velora release build failed. Upload Velora-gradle-failure.txt or copy the LAST GRADLE LINES shown above.')

release_dir = project / 'app' / 'build' / 'outputs' / 'apk' / BUILD_CHANNEL / 'release'
release_apks = sorted(
    release_dir.glob('*.apk'),
    key=lambda p: p.stat().st_mtime,
)
if not release_apks:
    raise RuntimeError('Release APK was not found')

release_apk = pathlib.Path('/content/Velora-release.apk')
shutil.copy2(release_apks[-1], release_apk)

apksigner = pathlib.Path('/content/android-sdk/build-tools/35.0.0/apksigner')
verify = subprocess.run([
    str(apksigner), 'verify', '--min-sdk-version', '26', '--verbose', '--print-certs', str(release_apk)
], capture_output=True, text=True)
print(verify.stdout)
print(verify.stderr)
if verify.returncode != 0:
    raise RuntimeError('APK signature verification failed')

verify_text = verify.stdout + verify.stderr
print('APK signature verification: PASS')
print('Velora minSdk verification : 26')
print('Signature scheme details above are informational; apksigner exit status is authoritative for this build.')

sha256 = hashlib.sha256(release_apk.read_bytes()).hexdigest()
pathlib.Path('/content/Velora-release.apk.sha256').write_text(
    sha256 + '  Velora-release.apk\n'
)

print('\nVELORA RELEASE READY')
print('Channel :', BUILD_CHANNEL)
print('APK     :', release_apk)
print('Size    :', f'{release_apk.stat().st_size / 1024 / 1024:.2f} MiB')
print('SHA-256 :', sha256)


## Download release
Install `Velora-release.apk` on the phone. The `.sha256` file is only for integrity checking.

In [ ]:
from google.colab import files
files.download('/content/Velora-release.apk')
files.download('/content/Velora-release.apk.sha256')
